# PolyWhisper Product Training — Kaggle 2×T4 (resumable)

Trains whisper-small + per-language LoRA experts (hi/ta/te/bn/mr) on IndicVoices-ST in parallel on 2 GPUs. Fully resumable: state + adapters persist to HF Hub every epoch; re-run this notebook to continue after a session death.

**Before running:** add Kaggle secret `HF_TOKEN` (your Hugging Face token — needed for gated IndicVoices-ST dataset + persisting to `eulogik/polywhisper`).

In [ ]:
# 1. Verify GPU + torch
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('gpu count:', torch.cuda.device_count())
if not torch.cuda.is_available():
    print('Installing CUDA torch (cu121)...')
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--index-url', 'https://download.pytorch.org/whl/cu121',
                    'torch', 'torchvision', 'torchaudio'], check=True)
    print('Restart runtime after install!')
else:
    print('GPU OK')

In [ ]:
# 2. Deps + secrets
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'datasets', 'soundfile', 'librosa', 'huggingface_hub'], check=True)

import os
from kaggle_secrets import UserSecretsClient
try:
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
# 3. Download training scripts from HF Hub (always latest)
from huggingface_hub import hf_hub_download
for f in ['train_v3.py', 'eval_lang_pure.py', 'kaggle_train_resumable.py']:
    p = hf_hub_download('eulogik/polywhisper', f, repo_type='model')
    os.rename(p, f)
    print('downloaded', f)
print('Scripts ready.')

In [ ]:
# 4. Run product training (2 GPUs in parallel, resumable, auto-eval, upload to HF)
!python kaggle_train_resumable.py

## After the run

Checkpoints + evals are on `eulogik/polywhisper` under `polywhisper_output_gpu0/` and `polywhisper_output_gpu1/`.

If the session dies: just re-run cells 1-4 — it restores state from HF and resumes exactly where it stopped.

Local step (on dev machine): copy adapters into `polywhisper_output/adapters_v3/`, re-run `normalize_ortho.py` + `make_results_table.py`.